In [1]:
def tubular_regularizer(covariance_matrices, eps=1e-6):
    """
    Computes tubular regularization loss based on eigenvalue ratios.
    Encourages one axis to be long and two axes to be small.
    
    Args:
        covariance_matrices: (N, 3, 3) tensor of covariance matrices
        eps: small constant for numerical stability
    
    Returns:
        Scalar loss value
    """
    # Compute eigenvalues for each covariance matrix
    eigenvalues = torch.linalg.eigvalsh(covariance_matrices)  # (N, 3)
    
    # Sort eigenvalues in ascending order: [λ_small1, λ_small2, λ_large]
    eigenvalues_sorted = torch.sort(eigenvalues, dim=-1)[0]
    
    lambda_short1 = eigenvalues_sorted[:, 0]
    lambda_short2 = eigenvalues_sorted[:, 1]
    lambda_long = eigenvalues_sorted[:, 2]
    print(f"Lambda short 1: {lambda_short1.mean().item():.6f}, Lambda short 2: {lambda_short2.mean().item():.6f}, Lambda long: {lambda_long.mean().item():.6f}")
    # Compute tubular loss: (λ_short1 + λ_short2) / λ_long
    # Minimizing this ratio encourages tubular shapes
    tubular_loss = (lambda_short1 + lambda_short2) / (lambda_long + eps)
    
    return tubular_loss.mean()

In [3]:
import torch
# test that function
if __name__ == "__main__":
    # Create dummy covariance matrices for testing
    N = 5
    cov_matrices = torch.rand(N, 3, 3)
    cov_matrices = cov_matrices @ cov_matrices.transpose(-1, -2)  # Make them symmetric positive semi-definite
    
    loss = tubular_regularizer(cov_matrices)
    print(f"Tube regularization loss: {loss.item():.6f}")

Lambda short 1: 0.030223, Lambda short 2: 0.251140, Lambda long: 2.977675
Tube regularization loss: 0.107291


In [4]:
# Estimate local tangent via PCA of nearby Gaussian centers
def estimate_tangents_pca(positions, k_neighbors=5):
    """
    Estimates local tangent direction for each point via PCA of k-nearest neighbors.
    
    Args:
        positions: (N, 3) tensor of Gaussian center positions
        k_neighbors: number of nearest neighbors to use for PCA
    
    Returns:
        (N, 3) tensor of tangent vectors
    """
    N = positions.shape[0]
    tangents = torch.zeros_like(positions)
    
    for i in range(N):
        # Compute distances to all other points
        dists = torch.norm(positions - positions[i:i+1], dim=-1)
        
        # Find k nearest neighbors (excluding self)
        _, nearest_indices = torch.topk(dists, k_neighbors + 1, largest=False)
        nearest_indices = nearest_indices[1:]  # Exclude self
        
        # Get neighbor positions
        neighbors = positions[nearest_indices]
        
        # Center the neighbors
        centered = neighbors - neighbors.mean(dim=0, keepdim=True)
        
        # Compute covariance matrix
        cov = (centered.T @ centered) / (k_neighbors - 1)
        
        # Get principal direction (largest eigenvector)
        eigenvalues, eigenvectors = torch.linalg.eigh(cov)
        tangents[i] = eigenvectors[:, -1]  # Principal direction
    
    return tangents

# test that function
if __name__ == "__main__":
    # Create dummy positions for testing
    N = 10
    positions = torch.rand(N, 3)
    
    tangents = estimate_tangents_pca(positions, k_neighbors=3)
    print(f"Tangent vectors:\n{tangents}")

Tangent vectors:
tensor([[-0.5584,  0.8139,  0.1606],
        [-0.3100,  0.8289,  0.4656],
        [-0.6718,  0.5604,  0.4844],
        [ 0.1784, -0.4032, -0.8975],
        [ 0.5448,  0.1334,  0.8279],
        [-0.7427,  0.5141, -0.4291],
        [-0.7834,  0.6190, -0.0552],
        [-0.5959,  0.0256, -0.8027],
        [-0.7727,  0.6246, -0.1133],
        [-0.1508,  0.1641, -0.9749]])
